In [1]:
# Calculate hyperfine line shift and broadening

In [4]:
import sys
sys.path.append("../src/")
sys.path.append("../src/AtomicTrit")

In [5]:
from AtomicTrit import spinexchange
from AtomicTrit import potentials
from AtomicTrit import dipolelosses
from AtomicTrit import constants
from AtomicTrit import elastic
import numpy as np
import pylab as plt

from spinexchange import SpinExChannels

In [ ]:
B_value  = 1e-5
potT     = potentials.Silvera_Triplet
potS     = potentials.Kolos_Singlet2_VDW

T_values = np.logspace(-3, 2, 20)


GVsT_H=[]
GVsT_T=[]

for c in SpinExChannels:
    
    GsH=[]
    GsT=[]

    for T in T_values:
        GsH.append(spinexchange.GetGFactor(c,  B_value, constants.HydrogenConstants(), T, potT,potS,np.linspace(1e-9,0.75,20000)))
        GsT.append(spinexchange.GetGFactor(c,  B_value, constants.TritiumConstants(), T, potT,potS,np.linspace(1e-9,0.75,20000)))
    GVsT_H.append(np.array(GsH))
    GVsT_T.append(np.array(GsT))


In [ ]:
channels = [r'bd->aa',r'cc->aa',r'cc->bd',r'bd->ac']
plt.figure(figsize = (8,6), dpi = 300)
for i,ch in enumerate(GVsT_H):
    plt.plot(dipolelosses.E_com(T_values), ch, label = channels[i])

for i,ch in enumerate(GVsT_T):
    plt.plot(dipolelosses.E_com(T_values), ch, label = channels[i], ls = '--')
plt.yscale('log')
plt.xscale('log')
plt.legend()

In [ ]:
def Lambda_0_DIS(rhos, p, l, mu):
    # l even
    delta_l_1 = elastic.GetPhaseShift(rhos, p, l, mu, potential=potentials.Silvera_Triplet, how_to_int='Radau')
    delta_l_0 = elastic.GetPhaseShift(rhos, p, l, mu, potential=potentials.Silvera_Singlet, how_to_int='Radau')
    return np.pi/(2*k**2) * (2*l+1)*np.sin(2*delta_l_1 - 2*delta_l_0)

def Lambda_1_DIS(rhos, p, l, mu):
    return 0
def Lambda_2_DIS(rhos, p, l, mu):
    return 0

def Sigma_0_DIS(rhos, p, l, mu):
    return 0

def Sigma_1_DIS(rhos, p, l, mu):
    delta_l_1 = elastic.GetPhaseShift(rhos, p, l, mu, potential=potentials.Silvera_Triplet, how_to_int='Radau')
    delta_l_0 = elastic.GetPhaseShift(rhos, p, l, mu, potential=potentials.Silvera_Singlet, how_to_int='Radau')
    return np.pi/(k**2) * (-1)**l (2*l+1)*np.sin(delta_l_0 - delta_l_1)**2

def Sigma_2_DIS(rhos, p, l, mu):
    # l odd
    delta_l_1 = elastic.GetPhaseShift(rhos, p, l, mu, potential=potentials.Silvera_Triplet, how_to_int='Radau')
    delta_l_0 = elastic.GetPhaseShift(rhos, p, l, mu, potential=potentials.Silvera_Singlet, how_to_int='Radau')
    return np.pi/(k**2) * (2*l+1)*np.sin(delta_l_0 - delta_l_1)**2

In [ ]:
#Calculate higher partial wave elastic cross sections
k_eV  = np.linspace(1e-4, 4*constants.hcInEVAngstrom, 20)
k_A = k_eV / constants.hcInEVAngstrom

r0       = 1e-9
intlimit = 100 * constants.BohrInAng/constants.hcInEVAngstrom
rhos = np.linspace(r0, intlimit, 10)

muH=constants.HydrogenConstants.mu
muT=constants.TritiumConstants.mu

even_ls = np.arange(0, 16, 2)
odd_ls = np.arange(1, 17, 2)
ls = np.arange(0, 16, 1)

Lambda_0_DIS_l = {}
Lambda_0_DIS_total = np.zeros_like(k_A)   # activate if you want Σ_l


for l in even_ls:
    lambda_T_partial = np.array([Lambda_0_DIS(rhos, k, l, muH)
        for k in k_eV
    ])
    Lambda_0_DIS_l[l] = lambda_T_partial
    Lambda_0_DIS_total   += Lambda_0_DIS_l[l]

In [ ]:


plt.figure(figsize=(5,4.7), dpi=250)
for l in Lambda_0_DIS_l.keys():
    plt.plot(k_A, np.array(Lambda_0_DIS_l[l])* constants.hcInEVAngstrom**2*constants.BohrInAng**-2,alpha=0.5,label=r'$\mathtt{l}$='+str(l))
plt.plot(k_A, np.array(Lambda_0_DIS_total) * constants.hcInEVAngstrom**2*constants.BohrInAng**-2, label="Total",color='black')
plt.plot(k[0:-5],XS_smooth[0:-5]*constants.BohrInAng**-2, ':',label="Al-Maaitah",color='dimgrey',linewidth=4)
plt.xlabel(r'Relative momentum $k\;[\mathrm{\AA}^{-1}]$')
plt.ylabel(r'Total cross-section, $\sigma$ (a.u.)$^2$')
plt.xlim(0, 4)
plt.ylim(0, 800)
plt.title("T-T triplet scattering")
plt.legend(ncol=2)
plt.savefig("./Plots/TTElastic.png",dpi=250,bbox_inches='tight')